# Irrigation Training - v2.19-TD3 (deterministic VDN actor + LayerNorm VDN critic)

**Algorithm switch: SAC -> TD3.** The critic is BYTE-IDENTICAL to the v2.16/v2.17/v2.18 SAC runs (`_V211FactorizedContinuousCritic` -- VDN sum over 130 per-cell Q-nets, twin-Q, LayerNorm). Only the actor changes: deterministic `_TD3SharedActor` (no entropy, no log_std), with the IDENTICAL feature pipeline (shared LeakyReLU MLP + 2x-1 recenter + agent-major reshape). Marker = 2.19; runner.py dispatches TD3 checkpoints automatically (marker>=2.185 AND no log_std -> TD3.load).

## Why TD3

v2.18 (SAC alpha=0.002 + late reinjection) **matched MPC** (mean yield 3785 = 99.3% of MPC 3810; mean waterlog 18.2; wet x1 136 vs MPC 130) -- a legitimate result. But at alpha=0.002 the critic calibration wobbled (q_pred_mean briefly negative at 100-125k). TD3 is the principled way to take entropy to zero:
- **Removes the entropy objective** that pinned the actor mean at the 6 mm action-centre (the tanh midpoint). Deterministic policy can reach the tanh=-1 boundary (0 mm).
- **Adds target-policy smoothing** (Fujimoto et al. 2018: target_policy_noise=0.2, clip=0.5) -- the explicit replacement for the smoothing entropy implicitly provided, and the stabiliser whose ABSENCE let v2.7 cascade.
- **Explicit exploration noise** (0.20->0.05 over 100k, floor held) replaces SAC's policy stochasticity.
- Keeps: VDN LayerNorm twin-Q critic, gamma=0.99, tau=0.005, asymmetric actor LR (5x), policy_delay=2, 250k steps, RAIN_REF=30, linear r6.

| wet-year (mean of 3 budgets) | v2.16-fixed | v2.17 | v2.18 | MPC |
|---|---|---|---|---|
| x1 median (mm) | 152 | 144 | 136 | 130 |
| waterlog days | 76 | 54 | 37 | 18 |
| water (mm) | 397 | 362 | 336 | 309 |

## Acceptance (decide on x1/waterlog + STABILITY)

PRIMARY: wet x1 median < 134 mm AND wet waterlog < 32 days (close v2.18's residual gap to MPC).
SECONDARY: wet water < 330 mm; mean yield >= 3780 (no dry regression vs v2.18 dry 3999).
STABILITY (the whole point of target smoothing): critic_loss < 100; q_pred_mean NEVER negative; |q_inflation_pct| < 80%. **TD3 should be CLEANER than v2.18's alpha=0.002 run.**
ATTRIBUTION: if TD3 reaches MPC x1 where SAC alpha=0.002 stalled at 136, that isolates the entropy mu-pin as the binding constraint.

## Colab note
Results persist to Google Drive via the Cell-1 mount + archive cell. T4 (~2-2.5h) or A100 (~30-55min).


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE_ROOT='/content/drive/MyDrive/thesis_v219_td3_runs'; os.makedirs(DRIVE_ROOT,exist_ok=True)
print('Drive mounted:',DRIVE_ROOT)


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0). TD3 needs the same stack as SAC.
import subprocess, sys, os
WORK='/content'; repo=os.path.join(WORK,'thesis')
if os.path.exists(repo): subprocess.run(['rm','-rf',repo],check=True)
subprocess.run(['git','clone','https://github.com/taratorbati/thesis.git',repo],check=True)
os.chdir(repo); sys.path.insert(0,repo)
subprocess.run(['pip','install','--quiet','stable-baselines3==2.6.0','gymnasium','wandb','pytest'],check=True)
import torch; print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY'); print('OK WANDB key.')
except Exception as e:
    try:
        import getpass; k=getpass.getpass('WANDB key (Enter to skip): ').strip()
        if k: os.environ['WANDB_API_KEY']=k; print('OK set.')
        else: print('Skipping WandB.')
    except Exception: print('Skipping WandB.')
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


In [ ]:
# Pre-flight: smoke tests + 1000-step TD3 pilot.
# The pilot specifically exercises the NEW TD3 code: TD3VDNPolicy, the
# deterministic _TD3SharedActor (self.mu replaced by Identity + mu_head),
# AsymmetricLRTD3, target-policy smoothing, and the runner's TD3 dispatch.
import subprocess, sys
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short']).returncode==0,'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short']).returncode==0,'CRITIC TESTS FAILED'
print('\n1000-step TD3 pilot...')
from src.rl.train_v219_td3 import train_td3_v219
m = train_td3_v219(seed=999, output_dir='/content/pilot', wandb_project=None,
                   total_timesteps=1000, explore_decay_steps=300)
# Verify the saved checkpoint is a TD3 actor (no log_std) the runner can dispatch.
import glob, zipfile, io, torch
ck = glob.glob('/content/pilot/td3_v219_seed999/*final*.zip')
if ck:
    with zipfile.ZipFile(ck[0]) as z:
        sd = torch.load(io.BytesIO(z.open('policy.pth').read()), map_location='cpu', weights_only=False)
    assert 'actor.log_std.weight' not in sd, 'BUG: TD3 actor unexpectedly has log_std!'
    assert 'actor.mu_head.weight' in sd, 'BUG: TD3 actor missing mu_head!'
    assert abs(float(sd['actor.obs_norm_marker'].item()) - 2.19) < 0.01, 'BUG: marker != 2.19'
    print('  Checkpoint sanity: deterministic actor (no log_std), mu_head present, marker=2.19. OK')
print('\nOK pre-flight passed. Proceed.')


In [ ]:
# Full 250k TD3 training. ~30-55 min A100 / ~2-2.5 h T4.
SEED = 0    # CHANGE per session
from src.rl.train_v219_td3 import train_td3_v219
model = train_td3_v219(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    target_policy_noise=0.2, target_noise_clip=0.5, policy_delay=2,
    explore_sigma_start=0.20, explore_sigma_end=0.05, explore_decay_steps=100_000,
)
print('Training complete.')


In [ ]:
import shutil, os, datetime
src=f'/content/thesis/results/rl/td3_v219_seed{SEED}'
dst=os.path.join(DRIVE_ROOT,f'td3_v219_seed{SEED}_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree(src,dst,ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to Drive:',dst)


In [ ]:
# Post-training 9-cell eval. runner.py auto-detects the TD3 checkpoint
# (marker 2.19 + no log_std -> TD3.load) and applies RAIN_REF=30.
# exp_rl.py has NO --output-dir (writes to its default runs dir); UTF-8 stdout
# fix prevents the Windows label crash.
import subprocess, sys, os
model_path=f'/content/thesis/results/rl/td3_v219_seed{SEED}/best_model/best_model.zip'
final_path=f'/content/thesis/results/rl/td3_v219_seed{SEED}/td3_v219_seed{SEED}_final.zip'
print('Evaluating BEST (perfect)...')
r=subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','perfect'],
    capture_output=True,text=True)
print(r.stdout[-1500:])
if r.returncode!=0: print('STDERR:',r.stderr[-2500:])
assert r.returncode==0,'PERFECT EVAL FAILED'
print('\nEvaluating BEST (noisy, robustness)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','noisy','--noise-seed','42'])
if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k, perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
        '--model',final_path,'--scenario','all','--budget','all','--forecast','perfect'])
print('\nNOTE: outputs under results/runs/<model-name>/.')

import shutil, datetime, os
dst=os.path.join(DRIVE_ROOT,'eval_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree('/content/thesis/results/runs',dst,dirs_exist_ok=True); print('Eval archived:',dst)


In [ ]:
# PRIMARY DIAGNOSTIC (decide on x1/waterlog). Auto-locate eval dir.
import pandas as pd, numpy as np, json, glob, os
cands=[d for d in glob.glob('/content/thesis/results/runs/*td3_v219*') if os.path.isdir(d)]
cands+=[d for d in glob.glob('/content/thesis/results/runs/*') if os.path.isdir(d) and glob.glob(os.path.join(d,'sac_perfect_det_wet*100*seed0.parquet'))]
OUT=next(iter(sorted(set(cands),key=os.path.getmtime,reverse=True)),None)
print('Eval dir:',OUT); assert OUT,'No eval dir found.'
def wet_agg(d):
    ys,ws,wls,x1s=[],[],[],[]
    for b in ['100pct','85pct','70pct']:
        pj=glob.glob(os.path.join(d,f'sac_perfect_det_wet*{b}*seed0.json')); pq=glob.glob(os.path.join(d,f'sac_perfect_det_wet*{b}*seed0.parquet'))
        if not pj or not pq: continue
        m=json.load(open(pj[0]))['final_metrics']; df=pd.read_parquet(pq[0])
        ys.append(m['yield_kg_ha']); ws.append(m.get('water_used_mm')); wls.append(m.get('waterlog_days_per_agent')); x1s.append(float(df['x1'].median()))
    f=lambda a: float(np.mean([v for v in a if v is not None])) if a else float('nan')
    return f(ys),f(ws),f(wls),f(x1s)
y,w,wl,x1=wet_agg(OUT)
print('='*60)
print(f'{"metric":<16s}{"v2.19":>8s}{"v2.18":>8s}{"MPC":>6s}{"target":>9s}')
print(f'{"x1 median":<16s}{x1:8.1f}{136:8.0f}{130:6.0f}{"< 134":>9s}')
print(f'{"waterlog days":<16s}{wl:8.1f}{37:8.0f}{18:6.0f}{"< 32":>9s}')
print(f'{"water (mm)":<16s}{w:8.0f}{336:8.0f}{309:6.0f}{"< 330":>9s}')
print(f'{"wet yield":<16s}{y:8.0f}{3687:8.0f}{3752:6.0f}{"(info)":>9s}')
print('\nPRIMARY:')
print(f'  x1 < 134 : {"PASS" if x1<134 else "FAIL"} ({x1:.1f})')
print(f'  wlog < 32: {"PASS" if wl<32 else "FAIL"} ({wl:.1f})')
def umean(d,s,b):
    pq=glob.glob(os.path.join(d,f'sac_perfect_det_{s}*{b}*seed0.parquet')); return float(pd.read_parquet(pq[0])['u'].mean()) if pq else float('nan')
print(f'\n  dry/100 u_mean={umean(OUT,"dry","100pct"):.2f}  wet/100 u_mean={umean(OUT,"wet","100pct"):.2f}')


In [ ]:
# STABILITY DIAGNOSTIC -- the headline TD3 question (did target smoothing help?).
import os
run_dir=f'/content/thesis/results/rl/td3_v219_seed{SEED}'
br=os.path.join(run_dir,'bias_ratio_log.csv')
if os.path.exists(br):
    import pandas as pd
    b=pd.read_csv(br); print(b.to_string(index=False))
    neg=(b['q_pred_mean']<0).any(); mx=b['q_inflation_pct'].abs().max()
    print(f'\n  q_pred_mean ever negative: {neg}  (v2.18 SAC: True -- TD3 should be False)')
    print(f'  max |q_inflation_pct|:     {mx:.1f}%  (v2.18 SAC: 180%% -- TD3 should be < 80%%)')
    cleaner = (not neg) and mx < 80
    print(f'  VERDICT: {"CLEANER than v2.18 -- target smoothing worked" if cleaner else "NOT cleaner -- investigate (still may have better x1)"}')
else:
    print('No bias_ratio_log.csv at', br)

# critic_loss trajectory from tensorboard (cascade check).
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    import glob
    runs=glob.glob(os.path.join(run_dir,'tensorboard','*'))
    if runs:
        ea=EventAccumulator(runs[0]); ea.Reload()
        if 'train/critic_loss' in ea.Tags()['scalars']:
            cl=ea.Scalars('train/critic_loss'); mx=max(e.value for e in cl)
            print(f'\n  max critic_loss = {mx:.2f}  (STABLE if < 100; v2.7 cascade hit 6.9e12)')
except Exception as e:
    print('tensorboard read skipped:', e)


In [ ]:
# Resume from checkpoint. Fill in the Drive run dir from the archive cell.
# SEED=0; STEP=100_000
# CKPT=f'/content/drive/MyDrive/thesis_v219_td3_runs/<run-dir>/td3_v219_seed{SEED}/checkpoints/td3_v219_seed{SEED}_{STEP}_steps.zip'
# from src.rl.train_v219_td3 import AsymmetricLRTD3
# from src.rl.networks_td3 import TD3VDNPolicy
# model=AsymmetricLRTD3.load(CKPT, custom_objects={'policy_class':TD3VDNPolicy})
# # model.learn(total_timesteps=..., reset_num_timesteps=False)
# # NOTE: action_noise not restored; recreate NormalActionNoise + ExplorationNoiseDecayCallback.
